# PrivQ Demo

**PrivQ** is a privacy-preserving SQL query transformation library. It rewrites SQL strings to reduce side-channel leakage from query observers and provides a differentially-private join-order hint.

This notebook walks through the public API:

1. **`orq`** &mdash; alphabetically sort `SELECT` columns to normalize column order
2. **`shrinkwrap`** / **`shrinkwrap_pad`** &mdash; apply structural padding to obscure query shape
3. **`privq_join`** / **`estimator`** &mdash; DP-noisy join reordering using a `Laplace(1/ε)` mechanism
4. **`transform`** &mdash; run the full three-stage pipeline in one call

## Setup

```bash
pip install PrivQ
```

In [1]:
import PrivQ as p

print(f"PrivQ version: {p.__version__}")
print(f"Public API:    {p.__all__}")

PrivQ version: 1.3.0
Public API:    ['orq', 'shrinkwrap', 'shrinkwrap_pad', 'truncated_laplace', 'dp_resize', 'privq_join', 'estimator', 'transform', 'TableSizeEstimator', 'laplace_sample', 'noisy_count', 'Table', 'QueryPlan', 'Filter', 'Join', 'Aggregate', 'SimulationReport', 'StepResult', 'FullObliviousPolicy', 'DPResizePolicy', 'OrqPolicy']


---
## 1. ORQ &mdash; column-order normalization

An observer watching a stream of `SELECT` statements can sometimes infer the writer's intent from the *order* of projected columns (e.g., the most "interesting" column listed first). `orq()` removes that channel by sorting the column list alphabetically (case-insensitive, original casing preserved).

In [2]:
before = "SELECT salary, name, age FROM employees"
after  = p.orq(before)

print("before:", before)
print("after: ", after)

before: SELECT salary, name, age FROM employees
after:  SELECT age, name, salary FROM employees


### Edge cases ORQ handles

ORQ is intentionally conservative &mdash; it only touches queries where reordering is safe and meaningful.

In [3]:
examples = [
    # case-insensitive sort, preserves original casing
    "SELECT Salary, name, Age FROM employees",
    # function calls treated as a single token
    "SELECT COUNT(*), AVG(salary) FROM employees",
    # aliases stay attached to their column
    "SELECT salary as s, name as n FROM employees",
    # lowercase keywords work too
    "select salary, name from employees",
    # SELECT * is left alone
    "SELECT * FROM employees",
    # single column has nothing to sort
    "SELECT name FROM employees",
    # non-SELECT queries pass through unchanged
    "INSERT INTO employees (name) VALUES ('Alice')",
]

for sql in examples:
    print(f"in : {sql}")
    print(f"out: {p.orq(sql)}")
    print()

in : SELECT Salary, name, Age FROM employees
out: SELECT Age, name, Salary FROM employees

in : SELECT COUNT(*), AVG(salary) FROM employees
out: SELECT AVG(salary), COUNT(*) FROM employees

in : SELECT salary as s, name as n FROM employees
out: SELECT name as n, salary as s FROM employees

in : select salary, name from employees
out: select name, salary from employees

in : SELECT * FROM employees
out: SELECT * FROM employees

in : SELECT name FROM employees
out: SELECT name FROM employees

in : INSERT INTO employees (name) VALUES ('Alice')
out: INSERT INTO employees (name) VALUES ('Alice')



---
## 2. ShrinkWrap &mdash; structural padding

An observer watching query text can also infer query *shape*: whether it has a `JOIN`, an aggregate, a subquery, a `WITH` clause, an `ORDER BY`, etc. `shrinkwrap()` adds dummy syntactic elements so two queries with different shapes look more alike on the wire.

Padding is controlled by `pad_level`:

| Level | What gets added |
|:----:|:----------------|
| `1` | Comment wrapper only |
| `2` | Level 1 + column, aggregate, CTE, `ORDER BY`, and subquery pads (**default**) |
| `3` | Level 2 + dummy `LEFT JOIN` |

Pads are *conditional*: e.g., the aggregate pad only fires if the query already has an aggregate, so we don't accidentally turn a non-aggregate query into one.

In [4]:
sql = "SELECT * FROM users"

for level in (1, 2, 3):
    print(f"--- pad_level={level} ---")
    print(p.shrinkwrap(sql, padding=True, pad_level=level))
    print()

--- pad_level=1 ---

    -- shrinkwrap padding
    SELECT * FROM users
    -- end padding


--- pad_level=2 ---

    -- shrinkwrap padding
    SELECT *, /* shrinkwrap */ 1 as pad_col /* end */ FROM users
    -- end padding


--- pad_level=3 ---

    -- shrinkwrap padding
    SELECT *, /* shrinkwrap */ 1 as pad_col /* end */ FROM users
    -- end padding
 LEFT JOIN (SELECT 1 as dummy) sw ON 1=1 /* shrinkwrap join */



### `padding=False` short-circuits everything

Useful for A/B comparisons or pipelines where you want padding controlled by a single flag.

In [5]:
assert p.shrinkwrap(sql, padding=False) == sql
assert p.shrinkwrap(sql, padding=False, pad_level=3) == sql
print("padding=False returns the input unchanged.")

padding=False returns the input unchanged.


### Applying a single pad with `shrinkwrap_pad`

When you want fine-grained control, `shrinkwrap_pad(sql, name)` applies one named pad. Available pads: `query`, `column`, `aggregate`, `cte`, `join`, `orderby`, `subquery`, `insert`, `update`, `delete`.

In [6]:
examples = {
    "query":     "SELECT 1",
    "column":    "SELECT * FROM users",
    "aggregate": "SELECT AVG(salary) as avg_salary FROM employees",
    "cte":       "WITH sales AS (SELECT * FROM orders) SELECT * FROM sales",
    "join":      "SELECT * FROM users",
    "orderby":   "SELECT * FROM users ORDER BY name",
    "subquery":  "SELECT * FROM (SELECT AVG(price) FROM products) sub",
    "insert":    "INSERT INTO users (name) VALUES ('Alice')",
    "update":    "UPDATE users SET name = 'Bob' WHERE id = 1",
    "delete":    "DELETE FROM users WHERE id = 1",
}

for pad, sql in examples.items():
    print(f"--- pad={pad!r} ---")
    print("in :", sql)
    print("out:", p.shrinkwrap_pad(sql, pad))
    print()

--- pad='query' ---
in : SELECT 1
out: 
    -- shrinkwrap padding
    SELECT 1
    -- end padding


--- pad='column' ---
in : SELECT * FROM users
out: SELECT *, /* shrinkwrap */ 1 as pad_col /* end */ FROM users

--- pad='aggregate' ---
in : SELECT AVG(salary) as avg_salary FROM employees
out: SELECT AVG(salary) as avg_salary, /* shrinkwrap */ SUM(0) as pad_agg /* end */ FROM employees

--- pad='cte' ---
in : WITH sales AS (SELECT * FROM orders) SELECT * FROM sales
out: WITH dummy_cte AS (SELECT 1 as pad_col), /* shrinkwrap */ sales AS (SELECT * FROM orders) SELECT * FROM sales

--- pad='join' ---
in : SELECT * FROM users
out: SELECT * FROM users LEFT JOIN (SELECT 1 as dummy) sw ON 1=1 /* shrinkwrap join */

--- pad='orderby' ---
in : SELECT * FROM users ORDER BY name
out: SELECT * FROM users ORDER BY 0 /* shrinkwrap */, name

--- pad='subquery' ---
in : SELECT * FROM (SELECT AVG(price) FROM products) sub
out: SELECT * FROM (SELECT AVG(price), /* shrinkwrap */ 0 as pad_col /* end */ FR

An unknown pad name raises `ValueError`:

In [7]:
try:
    p.shrinkwrap_pad("SELECT 1", "nonexistent")
except ValueError as e:
    print("ValueError:", e)

ValueError: Unknown pad 'nonexistent'. Choose from: aggregate, column, cte, delete, insert, join, orderby, query, subquery, update


---
## 3. DP Resize &mdash; bounding result cardinality with truncated Laplace

The ShrinkWrap paper (Bater et al., VLDB 2018) does more than syntactic padding: between operators it uses a **truncated Laplace mechanism** (Def. 4) to release a differentially-private upper bound on each intermediate result's cardinality, then truncates the secure array to that bound. `PrivQ` exposes this mechanism in two pieces:

- `truncated_laplace(epsilon, delta, sensitivity)` &mdash; the noise primitive itself
- `dp_resize(sql, true_count, epsilon, delta, sensitivity)` &mdash; rewrites SQL to append `LIMIT (true_count + noise)`

Unlike `laplace_sample`/`noisy_count`, the truncated Laplace is shifted right by an amount $\eta_0$ chosen so that $\Pr[\eta < \Delta_c] \le \delta$. The noisy bound therefore almost always exceeds the true cardinality, so real rows are essentially never dropped.

In [8]:
# Truncated Laplace draws cluster well above the sensitivity, by design.
import statistics

print(f"{'epsilon':>8}  {'delta':>8}  {'mean':>8}  {'stdev':>8}  {'min':>5}")
print("-" * 44)
for eps in (0.1, 0.5, 1.0, 5.0):
    draws = [p.truncated_laplace(eps, 1e-5, sensitivity=1.0) for _ in range(2000)]
    print(f"{eps:>8.2f}  {1e-5:>8.0e}  {statistics.mean(draws):>8.2f}  "
          f"{statistics.stdev(draws):>8.2f}  {min(draws):>5d}")

 epsilon     delta      mean     stdev    min
--------------------------------------------
    0.10     1e-05    108.11     13.74     52
    0.50     1e-05     22.02      2.90      7
    1.00     1e-05     11.23      1.49      4
    5.00     1e-05      2.17      0.40      0


### `dp_resize` &mdash; rewriting a SQL query at the text layer

`dp_resize` computes a noisy size bound and appends (or replaces) a trailing `LIMIT`. The DP guarantee applies to the released `LIMIT` value: any two neighboring databases differing by `sensitivity` rows produce LIMIT distributions within $e^\varepsilon$ (with failure probability $\delta$).

In [9]:
# Appending LIMIT when none is present
print(p.dp_resize("SELECT name FROM users WHERE active = 1",
                   true_count=100, epsilon=1.0, delta=1e-5))
print()

# Replacing an existing trailing LIMIT
print(p.dp_resize("SELECT name FROM users LIMIT 5",
                   true_count=100, epsilon=1.0, delta=1e-5))
print()

# Trailing semicolon is preserved
print(p.dp_resize("SELECT * FROM t;",
                   true_count=10, epsilon=1.0, delta=1e-5))
print()

# Inner-subquery LIMITs are left alone — they belong to a different operator
print(p.dp_resize("SELECT * FROM (SELECT * FROM t LIMIT 3) sub",
                   true_count=100, epsilon=1.0, delta=1e-5))

SELECT name FROM users WHERE active = 1 LIMIT 114

SELECT name FROM users LIMIT 111

SELECT * FROM t LIMIT 20;

SELECT * FROM (SELECT * FROM t LIMIT 3) sub LIMIT 111


### Privacy/utility tradeoff

Smaller $\varepsilon$ means more noise &mdash; the `LIMIT` overshoots the true cardinality by more on average, costing query performance (the downstream engine processes more dummy rows) but providing stronger privacy.

In [10]:
true_count = 1000
TRIALS     = 500

print(f"true_count = {true_count}")
print(f"{'epsilon':>8}  {'mean LIMIT':>12}  {'mean overhead':>14}")
print("-" * 40)
for eps in (0.01, 0.1, 0.5, 1.0, 5.0):
    limits = []
    for _ in range(TRIALS):
        out = p.dp_resize("SELECT * FROM t", true_count,
                          epsilon=eps, delta=1e-5)
        limits.append(int(out.rsplit("LIMIT", 1)[1].strip()))
    mean_limit = sum(limits) / len(limits)
    overhead   = (mean_limit - true_count) / true_count
    print(f"{eps:>8.2f}  {mean_limit:>12.1f}  {overhead:>13.1%}")

true_count = 1000
 epsilon    mean LIMIT   mean overhead
----------------------------------------
    0.01        2080.5         108.1%
    0.10        1107.6          10.8%
    0.50        1022.1           2.2%
    1.00        1011.1           1.1%
    5.00        1002.2           0.2%


---
## 4. PrivQ join &mdash; differentially-private join reordering

`privq_join(tables, catalog, epsilon)` reorders a list of table names smallest&rarr;largest using **noisy** size estimates. True row counts from the catalog are never stored &mdash; `Laplace(1/ε)` noise is added before insertion into an internal hash map + min-heap. The catalog can be loaded once and queried many times.

Smaller `ε` &rArr; more noise &rArr; stronger privacy, less accurate ordering.

In [11]:
catalog = [
    ("users",    100),
    ("orders",   200),
    ("products",  50),
]
tables  = ["orders", "users", "products"]

ordered = p.privq_join(tables, catalog=catalog, epsilon=0.5)
print("input order:  ", tables)
print("DP-noisy order:", ordered)

input order:   ['orders', 'users', 'products']
DP-noisy order: ['products', 'users', 'orders']


### Sweeping ε: trading privacy for accuracy

With a high-ε (low-noise) regime, the order tracks true sizes; with a low-ε (high-noise) regime, the order may be permuted. We measure how often the noisy order matches the true order across many trials.

In [12]:
true_order = ["products", "users", "orders"]  # 50, 100, 200
TRIALS     = 1000

print(f"{'epsilon':>8}  {'P(correct)':>12}")
print("-" * 24)
for eps in (0.01, 0.1, 0.5, 1.0, 10.0, 100.0):
    hits = sum(
        p.privq_join(tables, catalog=catalog, epsilon=eps) == true_order
        for _ in range(TRIALS)
    )
    print(f"{eps:>8.2f}  {hits/TRIALS:>12.3f}")

 epsilon    P(correct)
------------------------
    0.01         0.541
    0.10         0.999
    0.50         1.000
    1.00         1.000
   10.00         1.000
  100.00         1.000


### Reusing an estimator with `estimator()`

If you'll run many `join_order` queries against the same catalog, build the estimator once and reuse it &mdash; otherwise you're re-noising and rebuilding the heap on every call.

In [13]:
est = p.estimator(catalog, epsilon=0.5)
print("table_count:", est.table_count)
print("epsilon:    ", est.epsilon)
print()
print("join_order(['orders','users']):           ", est.join_order(["orders", "users"]))
print("join_order(['products','orders']):        ", est.join_order(["products", "orders"]))
print("join_order(['users','unknown_table']):    ", est.join_order(["users", "unknown_table"]),
      "   # unknown tables sort last")

table_count: 3
epsilon:     0.5

join_order(['orders','users']):            ['users', 'orders']
join_order(['products','orders']):         ['products', 'orders']
join_order(['users','unknown_table']):     ['users', 'unknown_table']    # unknown tables sort last


### Inspecting the noisy heap

`heap_snapshot()` returns the internal min-heap so you can see the noised sizes the estimator is using. (Real catalog row counts never leave the constructor.)

In [14]:
snap = est.heap_snapshot()

print(f"{'table':>10}  {'noisy_size':>12}")
print("-" * 26)
for entry in snap:
    print(f"{entry.table_name:>10}  {entry.noisy_size:>12}")

     table    noisy_size
--------------------------
  products            53
     users           106
    orders           200


### Low-level DP primitives

If you want to use the noise mechanism without the table-size machinery, `laplace_sample(scale)` and `noisy_count(true_count, epsilon)` are re-exported at the package level.

In [15]:
import statistics

samples = [p.laplace_sample(1.0) for _ in range(10_000)]
print(f"laplace_sample(scale=1.0) over 10k draws:")
print(f"  mean  = {statistics.mean(samples):.3f}   (expected ≈ 1.0, since output is |Lap(0,1)|)")
print(f"  stdev = {statistics.stdev(samples):.3f}")
print()

print("noisy_count(true=100, ε=0.5) over 10 draws:")
print(" ", [p.noisy_count(100, 0.5) for _ in range(10)])

laplace_sample(scale=1.0) over 10k draws:
  mean  = 1.003   (expected ≈ 1.0, since output is |Lap(0,1)|)
  stdev = 1.004

noisy_count(true=100, ε=0.5) over 10 draws:
  [102, 103, 100, 101, 105, 100, 103, 103, 101, 103]


---
## 5. `transform()` &mdash; the full pipeline

`transform()` runs all three stages in order and returns a dict with each stage's output. It does **not** rewrite the `FROM`/`JOIN` clause itself &mdash; the join order is returned as data, on the assumption that an upstream layer with a real SQL parser will apply it.

In [16]:
result = p.transform(
    "SELECT salary, name FROM orders JOIN users ON orders.user_id = users.id",
    catalog=[("users", 100), ("orders", 200)],
    epsilon=0.5,
    padding=True,
    pad_level=2,
)

for stage, value in result.items():
    print(f"--- {stage} ---")
    print(value)
    print()

--- original ---
SELECT salary, name FROM orders JOIN users ON orders.user_id = users.id

--- orq ---
SELECT name, salary FROM orders JOIN users ON orders.user_id = users.id

--- join_order ---
['users', 'orders']

--- shrinkwrap ---

    -- shrinkwrap padding
    SELECT name, salary FROM orders JOIN users ON orders.user_id = users.id
    -- end padding




### Without a catalog, the join stage is skipped

In [17]:
result = p.transform("SELECT salary, name FROM employees")
for stage, value in result.items():
    print(f"{stage:>10}: {value!r}")

  original: 'SELECT salary, name FROM employees'
       orq: 'SELECT name, salary FROM employees'
join_order: None
shrinkwrap: '\n    -- shrinkwrap padding\n    SELECT name, salary FROM employees\n    -- end padding\n'


### `Query` helper &mdash; thin wrapper for the OO-inclined

In [18]:
q = p.Query("SELECT b, a, c FROM t")
print("q:        ", q)
print("padded:   ", q.query(padding=True, pad_level=2)["shrinkwrap"])

q:         SELECT b, a, c FROM t
padded:    
    -- shrinkwrap padding
    SELECT a, b, c FROM t
    -- end padding



<!-- PRIVQ_PLAN_SECTION -->
---
## 5. Plan-layer simulator &mdash; comparing oblivious strategies

The previous sections operate on SQL strings. PrivQ also ships a **plan-layer
simulator** that lets you build a compositional query plan and run it under
three different padding strategies, exactly as the ORQ and Shrinkwrap papers
compare them:

- **`FullObliviousPolicy`** &mdash; the SMCQL-era baseline that pads every
  intermediate result to its worst-case Cartesian size
- **`DPResizePolicy(epsilon, delta)`** &mdash; Shrinkwrap's truncated-Laplace
  resize (Bater Def. 4 / Algorithm 1), applied per operator
- **`OrqPolicy`** &mdash; the ORQ join-aggregation bound (Baum &sect;3.3),
  which collapses naive `n*m` joins to `n+m` and uses `privq_join` for table
  ordering

The simulator never executes real MPC &mdash; it threads cardinalities
through the operator chain and reports a simulated I/O cost using the Bater
cost model (Table 2). It's a teaching artifact for the privacy/performance
tradeoff space.


In [19]:
from PrivQ.plan import (
    Table, QueryPlan, Filter, Join, Aggregate,
    FullObliviousPolicy, DPResizePolicy, OrqPolicy,
)


### Build a plan and attach tables

We model a healthcare-style query: filter `diagnosis` for heart-disease
patients, join in `medication`, join in `demographic`, then distinct on
patient id. (Same shape as Bater Figure 2's running example.)


In [20]:
plan = QueryPlan([
    Filter("diagnosis",  predicate="diag == 'hd'",  selectivity=0.01),
    Join("",             "medication",  on="code",  multiplicity=3),
    Join("",             "demographic", on="pid",   multiplicity=1),
    Aggregate(group_by="pid", op="distinct"),
])

tables = [
    Table("diagnosis",   rows=10_000),
    Table("medication",  rows=20_000),
    Table("demographic", rows= 5_000),
]


### Compare all three policies side by side

Each `simulate()` call returns a `SimulationReport` with per-operator true
cardinality, padded (observable) cardinality, and simulated cost.


In [21]:
for policy in [
    FullObliviousPolicy(),
    DPResizePolicy(epsilon=1.0, delta=1e-6),
    OrqPolicy(),
]:
    print(plan.simulate(tables=tables, policy=policy))
    print()


Policy: FullOblivious
Table order: ['diagnosis', 'medication', 'demographic']

  step                                  true             padded                cost
  ------------------------------------  ----  -----------------  ------------------
  Filter(diagnosis, diag == 'hd')        100             10,000              20,000
  Join(, medication, on=code)            300        200,000,000         400,010,000
  Join(, demographic, on=pid)            300  1,000,000,000,000   2,000,200,000,000
  Aggregate(group_by=pid, op=distinct)   300  1,000,000,000,000  80,726,274,277,298

  total cost: 82,726,874,307,298

Policy: DPResize
Table order: ['diagnosis', 'medication', 'demographic']

  step                                  true  padded       cost
  ------------------------------------  ----  ------  ---------
  Filter(diagnosis, diag == 'hd')        100     113     20,000
  Join(, medication, on=code)            300     340  4,520,113
  Join(, demographic, on=pid)            300     314

### The headline numbers

The three policies arrive at the same **true** cardinalities (which depend
only on the data) but radically different **padded** cardinalities &mdash;
and the cost model amplifies that gap because joins cost `O(n*m)`.

Putting it in a quick comparison table:


In [22]:
totals = {}
for policy in [
    FullObliviousPolicy(),
    DPResizePolicy(epsilon=1.0, delta=1e-6),
    OrqPolicy(),
]:
    rep = plan.simulate(tables=tables, policy=policy)
    totals[policy.name] = rep.total_cost

baseline = totals["FullOblivious"]
print(f"{'policy':<16} {'total cost':>22} {'speedup':>12}")
print("-" * 54)
for name, cost in totals.items():
    speedup = baseline / cost if cost else float("inf")
    print(f"{name:<16} {cost:>22,.0f} {speedup:>11,.0f}x")


policy                       total cost      speedup
------------------------------------------------------
FullOblivious        82,726,874,307,298           1x
DPResize                      8,015,958  10,320,274x
Orq                         701,151,656     117,987x


### Sweeping `epsilon` &mdash; the DP privacy/cost tradeoff

Lower `epsilon` means more privacy per release but more Laplace noise added
to each intermediate cardinality. Since cost scales with padded cardinality,
weaker privacy is faster. (At very low `epsilon`, the noise dwarfs the true
cardinality and we approach the fully-oblivious cost.)


In [23]:
print(f"{'epsilon':>10} {'avg total cost (20 runs)':>26}")
print("-" * 38)
for eps in (0.01, 0.1, 0.5, 1.0, 5.0):
    pol = DPResizePolicy(epsilon=eps, delta=1e-6)
    avg = sum(
        plan.simulate(tables=tables, policy=pol).total_cost for _ in range(20)
    ) / 20
    print(f"{eps:>10.2f} {avg:>26,.0f}")


   epsilon   avg total cost (20 runs)
--------------------------------------
      0.01                 97,343,124
      0.10                 16,409,898
      0.50                  8,936,319
      1.00                  7,992,971
      5.00                  7,245,208


### Scaling the input &mdash; cascading-join blowup

The reason Shrinkwrap and ORQ exist: under full-oblivious execution, the
cost of an `l`-join chain explodes as `n^(l+1)`. The simulator makes this
visible &mdash; we hold the plan fixed and grow the largest table.


In [24]:
print(f"{'rows':>10} {'Full-Oblivious':>20} {'DP Resize':>16} {'Orq':>14}")
print("-" * 64)
for n in (1_000, 5_000, 10_000, 50_000):
    rescaled_tables = [
        Table("diagnosis",   rows=n),
        Table("medication",  rows=2 * n),
        Table("demographic", rows=n // 2),
    ]
    full = plan.simulate(tables=rescaled_tables, policy=FullObliviousPolicy()).total_cost
    dp   = plan.simulate(tables=rescaled_tables, policy=DPResizePolicy(1.0, 1e-6)).total_cost
    orq  = plan.simulate(tables=rescaled_tables, policy=OrqPolicy()).total_cost
    print(f"{n:>10,} {full:>20,.0f} {dp:>16,.0f} {orq:>14,.0f}")


      rows       Full-Oblivious        DP Resize            Orq
----------------------------------------------------------------
     1,000       62,800,708,709          170,621      7,091,913
     5,000    9,590,934,299,663        2,242,814    175,540,828
    10,000   82,726,874,307,298        7,995,918    701,151,656
    50,000 12,082,245,355,977,608      179,785,560 17,506,570,949


### The simulator and the SQL layer compose

`DPResizePolicy` calls into the same C++ `truncated_laplace` primitive
that `p.dp_resize` uses, and `OrqPolicy.reorder_tables` calls `p.privq_join`.
Build the plan, decide the strategy, then if you need actual SQL out, hand
the per-operator true cardinalities into `p.dp_resize(sql, true_count, ...)`
to get a noised `LIMIT`.


In [25]:
import PrivQ as p

# Suppose the plan's first filter executes on a federated SQL engine.
# We can compute the DP-noised LIMIT for that operator's output:
sql = "SELECT pid FROM diagnosis WHERE diag = 'hd'"
true_count = 100  # what we learned from the simulator

print("plain SQL:")
print(" ", sql)
print()
print("with DP resize (epsilon=1.0, delta=1e-6):")
print(" ", p.dp_resize(sql, true_count, epsilon=1.0, delta=1e-6))


plain SQL:
  SELECT pid FROM diagnosis WHERE diag = 'hd'

with DP resize (epsilon=1.0, delta=1e-6):
  SELECT pid FROM diagnosis WHERE diag = 'hd' LIMIT 113


---
## Honest scope

PrivQ is a **string-level SQL rewriting** library. It is *not* a cryptographic MPC engine and does *not* execute queries obliviously over secret-shared data. Two things to keep in mind:

- **`shrinkwrap` / `shrinkwrap_pad` are syntactic** &mdash; they obscure query *shape* from a text-channel observer (presence of joins, aggregates, subqueries). They do not pad intermediate-result *cardinalities* inside an oblivious execution engine.
- **`dp_resize` faithfully implements the truncated Laplace mechanism** from Bater et al. Def. 4 and provides a real $(\varepsilon, \delta)$-DP bound on the released `LIMIT` value. Its guarantee applies to whoever sees the rewritten SQL string; downstream secure execution and side-channel padding are still the responsibility of the execution layer.
- **`privq_join` provides DP over the loaded catalog**, not per-query. Noise is sampled once at `load()` time and reused across `join_order` calls on that estimator instance. For a fresh $\varepsilon$-DP release per query, build a new `estimator()` each time.

PrivQ is intended to slot in front of a real secure execution layer &mdash; or to act as a teaching artifact for the ORQ + Shrinkwrap line of work.